# Qwen inference and submission

Load a pinned base and trained adapter, score every retained candidate using the training encoder, and apply the locked threshold independently. Export both required TSVs with every test reference, including empty rows. Validation checks exact candidate sets, external IDs, duplicate IDs, full test coverage and the match subset constraint. Files and the completion manifest publish only after validation.

In [ ]:
from pathlib import Path
import json
_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else next(
    p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "anshp" / "config.json").is_file())
_A = {"__file__": str(_ROOT / "00_setup_and_data_audit.ipynb")}
for _c in json.loads((_ROOT / "00_setup_and_data_audit.ipynb").read_text(encoding="utf-8"))["cells"]:
    if _c["cell_type"] == "code" and "er-definitions" in _c.get("metadata", {}).get("tags", []):
        exec("".join(_c["source"]), _A)

import csv
import itertools
import sqlite3
_T, _ = _A["load_helpers"](_ROOT / "03_qwen_training.ipynb")
_V, _ = _A["load_helpers"](_ROOT / "04_validation_and_thresholds.ipynb")
_B = _T["_B"]

def validate_submission(matching_path,candidate_path,candidate_dir):
    """Strict streaming check against the exact persisted pre-matcher candidates."""
    candidate_dir=Path(candidate_dir)
    cm=_A["require_manifest"](candidate_dir/"manifest.json","candidates")
    if cm["dataset"]!="test":raise ValueError("Submission requires test candidates")
    index_dir=Path(cm["index_dir"]);im=_A["require_manifest"](index_dir/"manifest.json","index")
    if im["fingerprint"]!=cm["identity"]["index"]:raise ValueError("Index changed")
    con=_B["sqlite_readonly"](candidate_dir/cm["database"])
    try:
        con.execute("ATTACH DATABASE ? AS idx",((index_dir/im["database"]).resolve().as_uri()+"?mode=ro",))
        for query,message in [
          ("""SELECT entity_id FROM idx.records WHERE source=1 EXCEPT
              SELECT reference_id FROM query_scope LIMIT 1""","Missing test references"),
          ("""SELECT reference_id FROM query_scope EXCEPT SELECT entity_id FROM idx.records
              WHERE source=1 LIMIT 1""","Unknown test references"),
          ("""SELECT c.candidate_id FROM candidates c LEFT JOIN idx.records r
              ON r.entity_id=c.candidate_id AND r.source IN (2,3)
              WHERE r.entity_id IS NULL LIMIT 1""","Invalid external ID"),
          ("""SELECT c.reference_id FROM candidates c LEFT JOIN query_scope q USING(reference_id)
              WHERE q.reference_id IS NULL LIMIT 1""","Candidate has no query scope")]:
            if con.execute(query).fetchone():raise ValueError(message)
        count=matches=pair_count=0
        def parse_ids(text):
            values=text.split(",") if text else []
            if len(values)!=len(set(values)) or any(not x or x!=x.strip() for x in values):
                raise ValueError("Empty, repeated or padded candidate ID")
            return set(values)
        with Path(matching_path).open(encoding="utf-8",newline="") as mf, \
             Path(candidate_path).open(encoding="utf-8",newline="") as cf:
            mr,cr=csv.reader(mf,delimiter="\t"),csv.reader(cf,delimiter="\t")
            if next(mr,None)!=["source1_entity_id","matched_entity_ids"] or \
               next(cr,None)!=["source1_entity_id","candidate_entity_ids"]:
                raise ValueError("Incorrect submission headers")
            expected=con.execute("SELECT reference_id FROM query_scope ORDER BY reference_id")
            for m,c,q in itertools.zip_longest(mr,cr,expected):
                if m is None or c is None or q is None or len(m)!=2 or len(c)!=2:
                    raise ValueError("Submission row-count/schema mismatch")
                if m[0]!=q[0] or c[0]!=q[0]:
                    raise ValueError("Missing, duplicated, unknown or out-of-order reference")
                predicted,candidates=parse_ids(m[1]),parse_ids(c[1])
                saved={r[0] for r in con.execute("SELECT candidate_id FROM candidates WHERE reference_id=?",(q[0],))}
                if candidates!=saved:raise ValueError("Output candidate set differs from matcher input")
                if not predicted<=candidates:raise ValueError("Match outside candidate set")
                count+=1;matches+=len(predicted);pair_count+=len(candidates)
        return {"references":count,"matches":matches,"candidate_pairs":pair_count,
                "complete_test_coverage":True,"exact_candidate_sets":True,"external_ids_valid":True}
    finally:con.close()

def export_submission(score_dir,policy_dir,output_dir):
    score_dir,policy_dir,out=Path(score_dir),Path(policy_dir),Path(output_dir)
    out.mkdir(parents=True,exist_ok=True)
    sm=_A["require_manifest"](score_dir/"manifest.json","scores")
    cd=Path(sm["candidate_dir"]);cm=_A["require_manifest"](cd/"manifest.json","candidates")
    pm=_A["require_manifest"](policy_dir/"manifest.json","policy")
    locked=json.loads((policy_dir/pm["policy_file"]).read_text(encoding="utf-8"))
    if cm["dataset"]!="test" or cm["fingerprint"]!=sm["identity"]["candidates"]:
        raise ValueError("Submission requires matching test score/candidate artifacts")
    if _V["policy_contract"](sm,cm)!=locked["contract"]:
        raise ValueError("Scorer/retriever differs from locked threshold policy")
    # Validate policy schema even if the entire candidate set is empty.
    _V["apply_match_policy"]([],["policy_check"],locked["policy"])
    con=_B["sqlite_readonly"](score_dir/sm["database"])
    uid=_A["uuid"].uuid4().hex
    mt,ct=out/f"matching-{uid}.tmp.tsv",out/f"candidates-{uid}.tmp.tsv"
    try:
        con.execute("ATTACH DATABASE ? AS cdb",((cd/cm["database"]).resolve().as_uri()+"?mode=ro",))
        for sql in (
            "SELECT reference_id,candidate_id FROM cdb.candidates EXCEPT SELECT reference_id,candidate_id FROM scores LIMIT 1",
            "SELECT reference_id,candidate_id FROM scores EXCEPT SELECT reference_id,candidate_id FROM cdb.candidates LIMIT 1"):
            if con.execute(sql).fetchone():raise ValueError("Incomplete/extra candidate scores")
        cursor=con.execute("""SELECT q.reference_id,c.candidate_id,s.score
            FROM cdb.query_scope q LEFT JOIN cdb.candidates c USING(reference_id)
            LEFT JOIN scores s ON s.reference_id=c.reference_id AND s.candidate_id=c.candidate_id
            ORDER BY q.reference_id,c.rank""")
        with mt.open("w",encoding="utf-8",newline="") as mf,ct.open("w",encoding="utf-8",newline="") as cf:
            mw,cw=csv.writer(mf,delimiter="\t",lineterminator="\n"),csv.writer(cf,delimiter="\t",lineterminator="\n")
            mw.writerow(["source1_entity_id","matched_entity_ids"])
            cw.writerow(["source1_entity_id","candidate_entity_ids"])
            for qid,group in itertools.groupby(cursor,key=lambda r:r[0]):
                rows=[r for r in group if r[1] is not None]
                predicted=_V["apply_match_policy"](rows,[qid],locked["policy"])[qid]
                ordered=[r[1] for r in rows]
                mw.writerow([qid,",".join(c for c in ordered if c in predicted)])
                cw.writerow([qid,",".join(ordered)])
        validation=validate_submission(mt,ct,cd)
        mp,cp=out/"matching_results.tsv",out/"candidate_pairs.tsv"
        mt.replace(mp);ct.replace(cp)
        result={"stage":"submission","complete":True,
            "fingerprint":_A["stable_hash"]({"scores":sm["fingerprint"],"policy":pm["fingerprint"]}),
            "files":[_A["file_entry"](mp,out),_A["file_entry"](cp,out)],
            "validation":validation,"score_fingerprint":sm["fingerprint"],
            "policy_fingerprint":pm["fingerprint"]}
        _A["atomic_json"](out/"manifest.json",result)
        return result
    finally:con.close()

def run_test_inference(model_dir,candidate_dir,policy_dir,score_dir,output_dir):
    model_dir,candidate_dir=Path(model_dir),Path(candidate_dir)
    mm=_A["require_manifest"](model_dir/"manifest.json","model")
    _A["load_helpers"](_ROOT/"03_qwen_training.ipynb",mm["qwen_helpers_sha256"])
    cm=_A["require_manifest"](candidate_dir/"manifest.json","candidates")
    im=_A["require_manifest"](Path(cm["index_dir"])/"manifest.json","index")
    if cm["dataset"]!="test" or im["identity"]["normalized"]!=mm["normalization_fingerprint"]:
        raise ValueError("Model normalization or test dataset mismatch")
    model,tokenizer,yes_no=_T["load_qwen"](mm["config"],adapter_dir=model_dir/mm["adapter"])
    _T["score_candidates"](model,tokenizer,yes_no,candidate_dir,score_dir,mm["config"],mm["fingerprint"])
    return export_submission(score_dir,policy_dir,output_dir)


In [ ]:
from pathlib import Path
import json
PROJECT_DIR = next(p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "anshp" / "config.json").is_file())
DATA_DIR = PROJECT_DIR.parent / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
CONFIG = json.loads((PROJECT_DIR / "config.json").read_text(encoding="utf-8"))

RUN_TEST_INFERENCE = False
if RUN_TEST_INFERENCE:
    result = run_test_inference(ARTIFACT_DIR/"model", ARTIFACT_DIR/"candidates_test",
        ARTIFACT_DIR/"policy", ARTIFACT_DIR/"scores_test", PROJECT_DIR/"output")
    print(result["validation"])
